# COLETA PROPOSIÇÕES

In [ ]:
import pandas as pd

def gerar_arquivao_proposicoes(anos):
    frames_autores = []
    frames_projetos = []

    for ano in anos:
        print(f"📥 Baixando projetos e autores de {ano}...")
        url_autores = f"https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-{ano}.csv"
        url_projetos = f"https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-{ano}.csv"

        try:

            df_autores = pd.read_csv(url_autores, sep=';', low_memory=False)
            df_autores = df_autores.dropna(subset=['idDeputadoAutor'])
            frames_autores.append(df_autores[['idProposicao', 'idDeputadoAutor', 'nomeAutor']])


            df_projetos = pd.read_csv(url_projetos, sep=';', low_memory=False)


            if 'keywords' not in df_projetos.columns:
                df_projetos['keywords'] = None


            df_projetos = df_projetos[df_projetos['siglaTipo'].isin(['PL', 'PEC'])]


            frames_projetos.append(df_projetos[['id', 'ultimoStatus_idSituacao', 'keywords']])

        except Exception as e:
            print(f" Erro no ano {ano}: {e}")

    print("\n Cruzando os dados...")
    tudo_autores = pd.concat(frames_autores, ignore_index=True)
    tudo_projetos = pd.concat(frames_projetos, ignore_index=True)
    tudo_projetos = tudo_projetos.rename(columns={'id': 'idProposicao'})

    df_completo = pd.merge(tudo_autores, tudo_projetos, on='idProposicao', how='inner')

    # =================================================================
    # KEYWORDS
    # =================================================================
    print("Extraindo arquivo de Keywords...")
    df_keywords = df_completo[['idDeputadoAutor', 'idProposicao', 'keywords']].dropna(subset=['keywords'])
    df_keywords = df_keywords.rename(columns={'idDeputadoAutor': 'deputado_id'})
    df_keywords.to_csv("modulo1_keywords.csv", index=False, encoding='utf-8')
    print(" Arquivo secundário gerado: 'modulo1_keywords.csv'")

    # =================================================================
    # RANKING NORMAL
    # =================================================================
    total_projetos = df_completo.groupby('idDeputadoAutor').agg(
        total_proposicoes=('idProposicao', 'size'),
        nomeAutor=('nomeAutor', 'first')
    ).reset_index()

    df_completo['ultimoStatus_idSituacao'] = pd.to_numeric(df_completo['ultimoStatus_idSituacao'], errors='coerce')
    df_aprovados = df_completo[df_completo['ultimoStatus_idSituacao'] == 1140]

    aprovados_count = df_aprovados.groupby('idDeputadoAutor').size().reset_index(name='proposicoes_aprovadas')

    ranking = pd.merge(total_projetos, aprovados_count, on='idDeputadoAutor', how='left')
    ranking['proposicoes_aprovadas'] = ranking['proposicoes_aprovadas'].fillna(0)

    ranking['taxa_aprovacao_%'] = (ranking['proposicoes_aprovadas'] / ranking['total_proposicoes']) * 100
    ranking['taxa_aprovacao_%'] = ranking['taxa_aprovacao_%'].round(2)

    ranking = ranking.rename(columns={'idDeputadoAutor': 'deputado_id', 'nomeAutor': 'deputado_nome'})

    return ranking


anos_historicos = list(range(2010, 2027))
df_modulo1 = gerar_arquivao_proposicoes(anos_historicos)

df_modulo1.to_csv("modulo1_proposicoes_completa.csv", index=False)
print(" Módulo 1 Finalizado! Arquivo principal gerado.")